In [16]:
import pandas as pd
import os

Load the DataFrames and add "season", "competition" columns

In [17]:
# Function to load league data from a given path and competition name
def load_league(path, competition_name):
    dfs = []

    for file in os.listdir(path):
        if file.endswith('.csv'):
            df = pd.read_csv(os.path.join(path, file))

            # season parse
            # ex: serie_a_20-21.csv
            season_part = file.split('_')[1]
            start, end = season_part.split('-')
            season = f"20{start}-20{end}"

            df['season'] = season
            df['competition'] = competition_name

            dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)


# Load all leagues
serie_a = load_league("../data/raw/serie-a", "Serie_A")
premier_league = load_league("../data/raw/premier-league", "Premier_League")
bundesliga = load_league("../data/raw/bundesliga", "Bundesliga")
la_liga = load_league("../data/raw/la-liga", "La_Liga")
ligue_1 = load_league("../data/raw/ligue-1", "Ligue_1")
super_lig = load_league("../data/raw/super-lig", "Super_Lig")
champions_league = load_league("../data/raw/champions-league", "Champions_League")
print(serie_a.head())

   Match Number  Round Number              Date               Location  \
0             1             1  19/09/2020 18:00        Artemio Franchi   
1             2             1  19/09/2020 20:45  Marcantonio Bentegodi   
2             3             1  20/09/2020 12:30          Ennio Tardini   
3             4             1  20/09/2020 15:00         Luigi Ferraris   
4             5             1  20/09/2020 18:00          Mapei Stadium   

       Home Team Away Team Result     season competition  
0     Fiorentina    Torino  1 - 0  2020-2021     Serie_A  
1  Hellas Verona      Roma  0 - 0  2020-2021     Serie_A  
2          Parma    Napoli  0 - 2  2020-2021     Serie_A  
3          Genoa   Crotone  4 - 1  2020-2021     Serie_A  
4       Sassuolo  Cagliari  1 - 1  2020-2021     Serie_A  


Renaming columns

In [18]:
dfs = [serie_a, premier_league, bundesliga, la_liga, ligue_1, super_lig, champions_league]

# Standardize column names
rename_dict = {
    "Home Team": "home_team",
    "Away Team": "away_team",
    "Date": "date",
    "Match Number": "match_number",
    "Round Number": "stage",
    "Result": "result"
}

# Rename columns and drop unnecessary ones
for df in dfs:
    df.rename(columns=rename_dict, inplace=True)
    df.drop(columns=['Location', 'Group'], inplace=True, errors='ignore')

# Convert stage to string
for df in dfs:
    df["stage"] = df["stage"].astype(str)

print(serie_a.head())

   match_number stage              date      home_team away_team result  \
0             1     1  19/09/2020 18:00     Fiorentina    Torino  1 - 0   
1             2     1  19/09/2020 20:45  Hellas Verona      Roma  0 - 0   
2             3     1  20/09/2020 12:30          Parma    Napoli  0 - 2   
3             4     1  20/09/2020 15:00          Genoa   Crotone  4 - 1   
4             5     1  20/09/2020 18:00       Sassuolo  Cagliari  1 - 1   

      season competition  
0  2020-2021     Serie_A  
1  2020-2021     Serie_A  
2  2020-2021     Serie_A  
3  2020-2021     Serie_A  
4  2020-2021     Serie_A  


Normalizing and adding columns

In [19]:
final_columns = [
    'home_team',
    'away_team',
    'date',
    'competition',
    'stage',
    'season',
    'is_after_ucl',
    'result',
    'home_goals',
    'away_goals',
    'points',
    'ucl_format',
    'home_team_match_count',
    'away_team_match_count',
    'team_elo',
    'opponent_elo',
    'elo_diff'
]

def prepare_dataframe(df):
    for col in final_columns:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[final_columns]
    return df

dfs = [prepare_dataframe(df) for df in dfs]
serie_a, premier_league, bundesliga, la_liga, ligue_1, super_lig, champions_league = dfs

print(serie_a.columns)
print(serie_a.head())

Index(['home_team', 'away_team', 'date', 'competition', 'stage', 'season',
       'is_after_ucl', 'result', 'home_goals', 'away_goals', 'points',
       'ucl_format', 'home_team_match_count', 'away_team_match_count',
       'team_elo', 'opponent_elo', 'elo_diff'],
      dtype='object')
       home_team away_team              date competition stage     season  \
0     Fiorentina    Torino  19/09/2020 18:00     Serie_A     1  2020-2021   
1  Hellas Verona      Roma  19/09/2020 20:45     Serie_A     1  2020-2021   
2          Parma    Napoli  20/09/2020 12:30     Serie_A     1  2020-2021   
3          Genoa   Crotone  20/09/2020 15:00     Serie_A     1  2020-2021   
4       Sassuolo  Cagliari  20/09/2020 18:00     Serie_A     1  2020-2021   

  is_after_ucl result home_goals away_goals points ucl_format  \
0         <NA>  1 - 0       <NA>       <NA>   <NA>       <NA>   
1         <NA>  0 - 0       <NA>       <NA>   <NA>       <NA>   
2         <NA>  0 - 2       <NA>       <NA>   <NA>     

Merging the Sesons and load as CSV

In [20]:
all_leagues = pd.concat([serie_a, premier_league, bundesliga, la_liga, ligue_1, super_lig, champions_league], ignore_index=True)
all_leagues.to_csv("../data/processed/all_matches_raw.csv", index=False)
print(all_leagues.head())

       home_team away_team              date competition stage     season  \
0     Fiorentina    Torino  19/09/2020 18:00     Serie_A     1  2020-2021   
1  Hellas Verona      Roma  19/09/2020 20:45     Serie_A     1  2020-2021   
2          Parma    Napoli  20/09/2020 12:30     Serie_A     1  2020-2021   
3          Genoa   Crotone  20/09/2020 15:00     Serie_A     1  2020-2021   
4       Sassuolo  Cagliari  20/09/2020 18:00     Serie_A     1  2020-2021   

  is_after_ucl result home_goals away_goals points ucl_format  \
0          NaN  1 - 0        NaN        NaN    NaN        NaN   
1          NaN  0 - 0        NaN        NaN    NaN        NaN   
2          NaN  0 - 2        NaN        NaN    NaN        NaN   
3          NaN  4 - 1        NaN        NaN    NaN        NaN   
4          NaN  1 - 1        NaN        NaN    NaN        NaN   

  home_team_match_count away_team_match_count team_elo opponent_elo elo_diff  
0                   NaN                   NaN      NaN          NaN